In [ ]:
# CELL 1: import functions

from functions.Diretories_Downloads import dir_project, cif_download
from functions.Primary_Filter import primary_filter
from functions.Extraction_Phases import extrair_fracoes_fase

In [ ]:
# CELL 2: Cria o diretório do projeto

Project_Name = "projeto_1"
dir_, dir_ref = dir_project(Project_Name)

print(dir_)
print(dir_ref)

In [ ]:
# CELL 3: Download referências

dict_refs = {
    "Hematite": 1011240,       # alpha-Fe2O3
    "Magnetite": 9002318,      # Fe3O4
    "Maghemite": 1528611,      # gamma-Fe2O3
    "Goethite": 9003077,       # alpha-FeOOH
    "Wüstite": 1011119,        # FeO
    "Ferro Metálico": 9006610, # Fe
    # ----- Fases Adicionais de Ferro -----
    "Lepidocrocite": 9002931,  # gamma-FeOOH (Produto comum de ferrugem)
    "Akaganeite": 9013531,     # beta-FeOOH (Formada em ambientes com cloreto)
    "Siderite": 9000110,       # FeCO3 (Carbonato de ferro)
    "Pyrite": 9000595,         # FeS2 (Sulfeto de ferro / "ouro de tolo")
    "Ilmenite": 1011175        # FeTiO3 (Óxido de ferro-titânio frequente)
}


for Nome, cod_ID in dict_refs.items():
    cif_download(Nome, cod_ID, dir_ref)

In [ ]:
# CELL 4: Run Primary_Filter.py

input_drx_raw = f"inputs/hematite_raw.txt"
df, amostra_norm, theta = primary_filter(correlation="pearson", input_file=input_drx_raw, ref_dir=dir_ref)
print(df)

In [ ]:
# CELL 5: Plot

for i, row in df.iterrows():
    print(f"Referência: {row['nome']}, Score: {row['score']:.2%}")
    score = row['score']
    nome = row['nome']
    ref_int_norm = row['ref_int_norm']

    # Plot
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(theta, amostra_norm, label="Amostra (experimental)", linewidth=0.8)
    ax.plot(theta, ref_int_norm, label=f"Referência: {nome} (score={score:.2%})", linewidth=0.8, alpha=0.8)
    ax.set_xlabel("2θ (°)")
    ax.set_ylabel("Intensidade Normalizada")
    ax.set_title("XRD: Amostra vs Melhor Referência")
    ax.legend()
    plt.tight_layout()
    plt.show()   

    break

In [ ]:
# CELL 6: Run Workflow_GSAS-II.py

import importlib
import functions.Workflow_GSAS2 as wf
importlib.reload(wf)
from functions.Workflow_GSAS2 import refinamento_sequencial_oxidos

refs_cif = [f"projects/{Project_Name}/ref/{df['nome'][0]}",
            f"projects/{Project_Name}/ref/{df['nome'][1]}",
            f"projects/{Project_Name}/ref/{df['nome'][2]}",
            ]

input_inst = "inputs/inst_rruff.prm"
resultado = refinamento_sequencial_oxidos(input_drx_raw, input_inst, refs_cif, Project_Name)

In [ ]:
# CELL 7: Plot do Refinamento de Rietveld

import matplotlib.pyplot as plt

x = resultado["x"]
yobs = resultado["yobs"]
ycalc = resultado["ycalc"]
diff = resultado["diff"]
wR = resultado["wR"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), height_ratios=[3, 1],
                                sharex=True, gridspec_kw={"hspace": 0.05})

# Painel superior: observado vs calculado
ax1.plot(x, yobs, "k-", label="Observado", linewidth=0.6)
ax1.plot(x, ycalc, "r-", label="Calculado", linewidth=0.6)
ax1.set_ylabel("Intensidade")
ax1.set_title(f"Refinamento de Rietveld — wR = {wR:.2f}%")
ax1.legend()

# Painel inferior: diferença
ax2.plot(x, diff, "b-", linewidth=0.5)
ax2.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax2.set_xlabel("2θ (°)")
ax2.set_ylabel("Diferença")

plt.tight_layout()
plt.show()

In [ ]:
# CELL 8: Extração automática dos percentuais de fase do .lst

lst_file_path = f"projects/{Project_Name}/results/{Project_Name}.lst"

dados_extraidos = extrair_fracoes_fase(lst_file_path)

print("--- Resultados do Refinamento ---")
if isinstance(dados_extraidos, dict):
    for fase, fracoes in dados_extraidos.items():
        print(f"\nFase: {fase}")
        print(f"  Fração da Fase (Phase Fraction): {fracoes['Phase Fraction']}")
        
        # Converte a Fração em Peso para porcentagem (x 100) para facilitar a leitura
        weight_pct = fracoes['Weight Fraction'] * 100
        print(f"  Fração em Peso (Weight Fraction): {fracoes['Weight Fraction']} ({weight_pct:.2f}%)")
else:
    print(dados_extraidos)